# SentimentScope: Transformer-Based IMDB Sentiment Analysis

**Udacity Masters in AI — Course 5 Project**

Build and train a custom transformer model (`DemoGPT`) for binary sentiment classification on the IMDB movie review dataset. The model must achieve >75% test accuracy.

---

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer
from datasets import load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import math
import os
from tqdm import tqdm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def get_device():
    """Select the best available device: CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print("Apple Silicon GPU (Metal Performance Shaders)")

## 2. Load and Explore the Dataset

### 2.1 Load the IMDB Dataset

In [ ]:
def load_imdb_data():
    """Load the IMDB dataset using HuggingFace datasets library."""
    dataset = load_dataset("imdb")
    return dataset

dataset = load_imdb_data()
print(dataset)

train_data = dataset["train"]
test_data = dataset["test"]

print(f"\nTraining samples: {len(train_data)}")
print(f"Test samples:     {len(test_data)}")

# Verify dimensions
assert len(train_data) == 25000, "Training set should have 25,000 samples"
assert len(test_data) == 25000, "Test set should have 25,000 samples"
print("\n✓ Dataset dimensions verified.")


### 2.2 Explore the Dataset

In [ ]:
# Convert to DataFrame for exploration
train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

# Descriptive statistics
train_df["review_length"] = train_df["text"].apply(len)
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))

print("=== Descriptive Statistics ===")
print(f"\nLabel distribution (train):")
print(train_df["label"].value_counts().rename({0: "Negative", 1: "Positive"}))
print(f"\nReview length (characters):")
print(train_df["review_length"].describe().round(1))
print(f"\nWord count:")
print(train_df["word_count"].describe().round(1))


### 2.3 Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Visualization 1: Sentiment distribution
sentiment_counts = train_df["label"].value_counts().sort_index()
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(["Negative (0)", "Positive (1)"], sentiment_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Sentiment Label Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(sentiment_counts.values):
    axes[0].text(i, v + 200, str(v), ha="center", fontweight="bold")

# Visualization 2: Word count distribution by sentiment
for label, color, name in [(0, "#e74c3c", "Negative"), (1, "#2ecc71", "Positive")]:
    subset = train_df[train_df["label"] == label]["word_count"]
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=name, edgecolor="black")
axes[1].set_title("Word Count Distribution by Sentiment", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].set_xlim(0, 1500)

# Visualization 3: Box plot of review lengths
data_to_plot = [
    train_df[train_df["label"] == 0]["word_count"],
    train_df[train_df["label"] == 1]["word_count"]
]
bp = axes[2].boxplot(data_to_plot, labels=["Negative", "Positive"], patch_artist=True)
bp["boxes"][0].set_facecolor("#e74c3c")
bp["boxes"][1].set_facecolor("#2ecc71")
axes[2].set_title("Review Length by Sentiment", fontsize=14, fontweight="bold")
axes[2].set_ylabel("Word Count")

plt.tight_layout()
plt.savefig("dataset_exploration.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: dataset_exploration.png")


### 2.4 Split Training Data into Train/Validation Sets

In [ ]:
from sklearn.model_selection import train_test_split

train_texts = train_data["text"]
train_labels = train_data["label"]

# 80/20 split, stratified
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=SEED, stratify=train_labels
)

test_texts = test_data["text"]
test_labels = test_data["label"]

print(f"Training set:   {len(train_texts)} samples")
print(f"Validation set: {len(val_texts)} samples")
print(f"Test set:       {len(test_texts)} samples")

# Verify stratification
print(f"\nTrain pos ratio: {sum(train_labels)/len(train_labels):.3f}")
print(f"Val pos ratio:   {sum(val_labels)/len(val_labels):.3f}")


## 3. Implement a Custom Dataset & DataLoader

### 3.1 Custom IMDBDataset Class

In [ ]:
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 256  # Truncate/pad to this length

class IMDBDataset(Dataset):
    """Custom PyTorch Dataset for IMDB reviews with BERT tokenization."""

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),        # (max_len,)
            "attention_mask": encoding["attention_mask"].squeeze(0),  # (max_len,)
            "label": torch.tensor(label, dtype=torch.long)
        }


### 3.2 Instantiate Datasets & DataLoaders

In [ ]:
# Create dataset objects
train_dataset = IMDBDataset(train_texts, train_labels, tokenizer)
val_dataset = IMDBDataset(val_texts, val_labels, tokenizer)
test_dataset = IMDBDataset(test_texts, test_labels, tokenizer)

# Verify lengths
assert len(train_dataset) == 20000, f"Expected 20000, got {len(train_dataset)}"
assert len(val_dataset) == 5000, f"Expected 5000, got {len(val_dataset)}"
assert len(test_dataset) == 25000, f"Expected 25000, got {len(test_dataset)}"

# Verify data types
sample = train_dataset[0]
assert sample["input_ids"].dtype == torch.long, "input_ids should be long"
assert sample["attention_mask"].dtype == torch.long, "attention_mask should be long"
assert sample["label"].dtype == torch.long, "label should be long"

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")
print(f"\nSample shapes:")
print(f"  input_ids:      {sample['input_ids'].shape}")
print(f"  attention_mask:  {sample['attention_mask'].shape}")
print(f"  label:           {sample['label'].shape}")
print("\n✓ All assertions passed.")


In [ ]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


## 4. Customize the Transformer Model

### 4.1 Model Architecture — DemoGPT

A custom transformer encoder for binary classification:
- Token + Positional Embeddings
- Multi-head Self-Attention layers
- Feed-forward network
- Mean pooling → Classification head

In [ ]:
class DemoGPT(nn.Module):
    """Custom Transformer model for binary sentiment classification."""

    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=512, max_len=MAX_LEN, num_classes=2, dropout=0.1):
        super(DemoGPT, self).__init__()

        self.d_model = d_model

        # Token and positional embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape

        # Create position indices
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)

        # Embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)

        # Create padding mask for transformer (True = ignore)
        if attention_mask is not None:
            src_key_padding_mask = (attention_mask == 0)
        else:
            src_key_padding_mask = None

        # Transformer encoder
        x = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        # Mean pooling (only over non-padded tokens)
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()  # (B, S, 1)
            x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
        else:
            x = x.mean(dim=1)

        # Classification
        logits = self.classifier(x)  # (B, num_classes)
        return logits


### 4.2 Verify Model Output Shape

In [ ]:
# Verify model output shape with dummy input
vocab_size = tokenizer.vocab_size
model = DemoGPT(vocab_size=vocab_size).to(device)

dummy_input = torch.randint(0, vocab_size, (4, MAX_LEN)).to(device)
dummy_mask = torch.ones(4, MAX_LEN, dtype=torch.long).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input, dummy_mask)

assert dummy_output.shape == (4, 2), f"Expected (4, 2), got {dummy_output.shape}"
print(f"Model output shape: {dummy_output.shape}")
print(f"✓ Output shape verified: (batch_size, num_classes) = {dummy_output.shape}")

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## 5. Implement Accuracy Calculation

In [ ]:
def calculate_accuracy(outputs, labels):
    """Calculate accuracy for binary classification.

    Args:
        outputs: Model logits of shape (batch_size, num_classes)
        labels:  Ground truth labels of shape (batch_size,)

    Returns:
        Accuracy as a float between 0 and 1.
    """
    predictions = torch.argmax(outputs, dim=1)
    correct = (predictions == labels).sum().item()
    accuracy = correct / labels.size(0)
    return accuracy

# Quick test
test_outputs = torch.tensor([[2.0, 0.1], [0.1, 3.0], [1.5, 0.5], [0.2, 1.8]])
test_labels = torch.tensor([0, 1, 0, 1])
test_acc = calculate_accuracy(test_outputs, test_labels)
assert test_acc == 1.0, f"Expected 1.0, got {test_acc}"
print(f"✓ calculate_accuracy verified: {test_acc}")


## 6. Train the Model

### 6.1 Training & Validation Functions

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """Train the model for one epoch."""
    model.train()
    total_loss = 0
    total_acc = 0
    num_batches = 0

    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
        total_acc += calculate_accuracy(outputs, labels)
        num_batches += 1

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / num_batches, total_acc / num_batches


def evaluate(model, dataloader, criterion, device):
    """Evaluate the model on validation/test data."""
    model.eval()
    total_loss = 0
    total_acc = 0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            total_acc += calculate_accuracy(outputs, labels)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


### 6.2 Run Training

In [ ]:
# Initialize model, optimizer, and loss
model = DemoGPT(vocab_size=tokenizer.vocab_size).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Training history
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}

print(f"Training DemoGPT for {NUM_EPOCHS} epochs...")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  Max seq length: {MAX_LEN}")
print(f"  Device:         {device}")
print("=" * 60)

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Step scheduler
    scheduler.step()

    # Save history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ★ New best model saved (val_acc={val_acc:.4f})")

print("\n" + "=" * 60)
print(f"Training complete. Best validation accuracy: {best_val_acc:.4f}")


### 6.3 Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss curves
ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#3498db", linewidth=2)
ax1.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#e74c3c", linewidth=2)
ax1.set_title("Training & Validation Loss", fontsize=14, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(epochs_range, history["train_acc"], "o-", label="Train Acc", color="#3498db", linewidth=2)
ax2.plot(epochs_range, history["val_acc"], "s-", label="Val Acc", color="#e74c3c", linewidth=2)
ax2.set_title("Training & Validation Accuracy", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")


## 7. Test and Evaluate the Model

Load the best model checkpoint and evaluate on the test set.

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pth", map_location=device))
print("Loaded best model checkpoint.")

# Evaluate on test set
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"\n{'='*60}")
print(f"  TEST RESULTS")
print(f"{'='*60}")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*60}")

# Rubric requirement: >75% test accuracy
assert test_acc > 0.75, f"Test accuracy {test_acc:.4f} is below 75% threshold!"
print(f"\n✓ Test accuracy ({test_acc*100:.2f}%) exceeds 75% threshold.")


## 8. Save Model Checkpoint

In [ ]:
# Save the final model checkpoint with all metadata
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "vocab_size": tokenizer.vocab_size,
        "d_model": 256,
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 512,
        "max_len": MAX_LEN,
        "num_classes": 2,
        "dropout": 0.1
    },
    "test_accuracy": test_acc,
    "training_history": history,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE
}

torch.save(checkpoint, "sentimentscope_checkpoint.pth")
print(f"Model checkpoint saved: sentimentscope_checkpoint.pth")
print(f"  Test accuracy: {test_acc*100:.2f}%")

# Simple inference interface
class SentimentPredictor:
    """Simple interface to load the model and perform inference."""

    def __init__(self, checkpoint_path, device=None):
        self.device = device or get_device()
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        config = checkpoint["config"]

        self.model = DemoGPT(**config).to(self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

    def predict(self, texts):
        """Predict sentiment for a list of texts."""
        if isinstance(texts, str):
            texts = [texts]

        encodings = self.tokenizer(
            texts, max_length=MAX_LEN, padding="max_length",
            truncation=True, return_tensors="pt"
        )

        with torch.no_grad():
            input_ids = encodings["input_ids"].to(self.device)
            attention_mask = encodings["attention_mask"].to(self.device)
            logits = self.model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

        results = []
        for i, text in enumerate(texts):
            sentiment = "Positive" if preds[i].item() == 1 else "Negative"
            confidence = probs[i][preds[i]].item()
            results.append({"text": text[:80] + "...", "sentiment": sentiment, "confidence": f"{confidence:.3f}"})
        return results

# Demo inference
predictor = SentimentPredictor("sentimentscope_checkpoint.pth")
demo_reviews = [
    "This movie was absolutely fantastic! Great acting and a wonderful storyline.",
    "Terrible film. Waste of time. The plot made no sense at all.",
    "An average movie, nothing special but not terrible either."
]
print("\n=== Demo Predictions ===")
for result in predictor.predict(demo_reviews):
    print(f"  {result['sentiment']} ({result['confidence']}) — {result['text']}")

## 9. Project Report

### Summary

This project implemented **SentimentScope**, a custom transformer-based model (`DemoGPT`) for binary sentiment analysis on the IMDB movie review dataset. The model was built from scratch using PyTorch's `nn.TransformerEncoder` with BERT subword tokenization, trained for 3 epochs on 20,000 training samples, and evaluated on the full 25,000-sample test set.

**Key Results:**
- The model achieved >75% test accuracy, meeting the project threshold
- Training utilized AdamW optimizer with cosine annealing learning rate scheduling
- Mean pooling over non-padded tokens proved effective for sequence-level classification

### Key Takeaways

1. **Subword tokenization matters**: Using BERT's subword tokenizer (`bert-base-uncased`) instead of character-level tokenization dramatically reduces sequence lengths and captures meaningful word fragments. This allows the model to handle out-of-vocabulary words gracefully and keep input sequences at a manageable 256 tokens while retaining most of the semantic content.

2. **Architecture adaptation for classification**: Converting a transformer from generation to classification requires fundamental changes — replacing next-token prediction with mean pooling + classification head. The key insight is that for classification, we need a single vector representation of the entire input, which we obtain by averaging all non-padded token embeddings before feeding into the classifier.

3. **Training stability techniques are essential**: Gradient clipping (`max_norm=1.0`), learning rate scheduling (cosine annealing), and weight decay (`0.01`) all contribute to stable training. Without these, transformer models can exhibit unstable loss curves and fail to converge, especially with the relatively small IMDB dataset.

### Suggestions for Improvement

- **Increase model capacity**: Larger `d_model`, more layers, or more attention heads could improve accuracy beyond 90%
- **Use pre-trained weights**: Fine-tuning a pre-trained BERT model would likely achieve 90%+ accuracy with fewer epochs
- **Data augmentation**: Techniques like back-translation or synonym replacement could improve generalization
- **Longer sequences**: Increasing `MAX_LEN` beyond 256 would capture more context from longer reviews